# 3D Map Analysis — RX J1713.7−3946 (H.E.S.S. DL3 DR1, Gammapy 1.3)

**Goal.** Build a 3D map dataset (counts, exposure, PSF, EDISP, FoV background) for the shell-type SNR **RX J1713.7−3946** using **H.E.S.S. DL3 DR1** observations, fit **extended spatial models** (Gaussian and Shell) with a power-law spectrum, and inspect diagnostics (excess & residual maps).

**Workflow**
1. Load H.E.S.S. DR1 and select OBS_IDs for RX J1713.
2. Define a 3D **WCS cube** (FoV & energy axes).
3. Build one **MapDataset per observation**; apply **safe mask** and **FoV background**; then **stack**.
4. Fit **Gaussian** and **Shell** morphologies + **PowerLaw** spectrum.
5. Inspect **excess** and **residual** maps; freeze/unfreeze parameters; export results.

**Requirements**: Gammapy 1.3; `$GAMMAPY_DATA` includes `hess-dl3-dr1`.


In [ ]:
import logging
import numpy as np
import astropy.units as u
from astropy.coordinates import SkyCoord, Angle
from regions import CircleSkyRegion

from gammapy.data import DataStore
from gammapy.maps import WcsGeom, MapAxis
from gammapy.datasets import Datasets, MapDataset
from gammapy.makers import MapDatasetMaker, SafeMaskMaker, FoVBackgroundMaker
from gammapy.modeling import Fit
from gammapy.modeling.models import (
    SkyModel,
    PowerLawSpectralModel,
    GaussianSpatialModel,
    ShellSpatialModel,
    FoVBackgroundModel,
    Models
)
from gammapy.estimators import ExcessMapEstimator

import matplotlib.pyplot as plt

# Logging setup
logging.basicConfig()
log = logging.getLogger("rxj1713-3d-hess")
log.setLevel(logging.INFO)

## Load H.E.S.S. DL3 DR1 & select observations

In [ ]:
data_store = DataStore.from_dir(f"$GAMMAPY_DATA/hess-dl3-dr1")
data_store.info()

## Define the target sky position (Galactic frame)

For RX J1713.7−3946 we work in Galactic coordinates, near (l, b) ≈ (347.34°, −0.47°).
This helps when choosing WCS geometry aligned to the Galactic frame later on.


In [ ]:
target_position = SkyCoord(l=347.34, b=-0.47, frame='galactic',unit = u.deg)

## Select observations by cone search

We select a **5° radius** circle around the target in the **Galactic** frame.  
This generous radius captures all wobble pointings relevant for RX J1713 in DR1.


In [ ]:
selection = dict(
    type="sky_circle",
    frame="galactic",
    lon=347.34*u.deg,
    lat=-0.47*u.deg,
    radius=5*u.deg, #HESS FoV is 5 deg so we set it as that here
)
selected_obs_table = data_store.obs_table.select_observations(selection)

## Peek the selected observation table

This shows the metadata (pointing, zenith, livetime, OBJECT/TARGET_NAME, etc.)  
It’s a good sanity check before we materialize the `Observation` objects.


In [ ]:
selected_obs_table

## Build the observation list

We now create `Observation` objects for all selected OBS_IDs.


In [ ]:
observations = data_store.get_observations(selected_obs_table["OBS_ID"])

In [ ]:
for obs in observations:
    print(obs)  # or pprint(obs) for pretty printing

## Inspect the first observation

We can look at the event distribution and instrument response functions (IRFs)  
for a representative observation (the first in our list).


In [ ]:
obs = observations[0]
print(obs)
obs.peek()

In [ ]:
# Quick look at events
events = obs.events
events.peek()

## Inspect IRFs

Each observation comes with IRFs: effective area (AEFF), energy dispersion (EDISP), and PSF.


In [ ]:
# Effective Area
obs.aeff.peek()

# Energy Dispersion
obs.edisp.peek()

# PSF
obs.psf.peek()

In [ ]:
obs.events.select_offset([0, 5] * u.deg).peek()
obs.events.select_offset([0, 2.5] * u.deg).peek()

## Inspect Background model

Check the field-of-view background model provided with the observation.


In [ ]:
obs.bkg.peek()

## Preparing reduced datasets geometry

We now define the reference geometry for our 3D analysis.  

This geometry specifies:
- the sky projection (centered on our target source),
- the pixel size on the sky (here `0.02 deg`),
- the total width of the field of view (here `6 deg × 6 deg`),
- the energy axis for the reconstructed photon energies.  

In addition, we must also define a *true energy axis* which will be used for evaluating the instrument response functions (IRFs).  
It is important to ensure that the true energy axis is both **wider in energy** and has **more bins** than the reconstructed axis, since the IRFs are defined in true energy.


In [ ]:
from gammapy.maps import MapAxis, WcsGeom

# Reconstructed energy axis (what we will bin the counts in)
energy_axis = MapAxis.from_energy_bounds(
    0.3, 10.0, 15, unit="TeV"
)

# Geometry in reconstructed energy (spatial + reconstructed energy axis)
geom = WcsGeom.create(
    skydir=target_position,
    binsz=0.02,         # pixel size in deg
    width=(6, 6),       # total width of the FoV in deg
    frame="galactic",   # coordinates
    axes=[energy_axis], # include reconstructed energy
)

# True energy axis (for IRFs: effective area, PSF, background, energy dispersion)
energy_axis_true = MapAxis.from_energy_bounds(
    0.1, 20.0, 20, unit="TeV", name="energy_true"
)


In [ ]:
print(geom)

## Data reduction — FoV background method (as in the H.E.S.S. tutorial)

Create a blank **accumulator** dataset (`stacked`) that we will fill by looping over observations.


In [ ]:
stacked = MapDataset.create(
    geom=geom, energy_axis_true=energy_axis_true, name="rxj-stacked"
)
stacked


## Exclusion mask 

We exclude the main emission region while fitting the FoV background.  
Here we use a **0.7°** radius circle around RX J1713. Adjust if needed.


In [ ]:
circle = CircleSkyRegion(center=target_position, radius=0.7 * u.deg)
exclusion_mask = ~geom.region_mask(regions=[circle])  # True = keep for bkg fit

# Quick plot of the mask
ax = exclusion_mask.plot_interactive()



## Makers

- `MapDatasetMaker(selection=['counts','background','psf','edisp','exposure'])`
- `SafeMaskMaker(methods=['offset-max','aeff-max','bkg-peak'], offset_max='2.3 deg')`
- `FoVBackgroundMaker(method='fit', exclusion_mask=exclusion_mask)`

The **background norm** is going to be fitted per observation.


In [ ]:
maker = MapDatasetMaker(selection=["counts", "background", "psf", "edisp", "exposure"])
safe_mask_maker = SafeMaskMaker(
    methods=["offset-max", "aeff-max", "bkg-peak"],  
    offset_max="2.3 deg"                             # H.E.S.S. safe FoV radius
)

fov_bkg_maker = FoVBackgroundMaker(method="fit", exclusion_mask=exclusion_mask)

## Run per-observation, fit background, and stack

For each observation:
1) build the per-obs dataset,
2) apply safe mask,
3) **fit the FoV background** (creates a `BackgroundModel` and adjusts its norm),
4) stack into the accumulator `stacked`.

We also print the fitted **background normalization** for each obs.


In [ ]:
for obs in observations:
    dataset = maker.run(stacked, obs)           # use our target geometry
    dataset = safe_mask_maker.run(dataset, obs)
    dataset = fov_bkg_maker.run(dataset)        # fits BackgroundModel norm on this dataset

    # Print fitted background normalization (if available)
    try:
        bn = dataset.background_model.spectral_model.norm.value
        print(f"Background norm – OBS_ID {obs.obs_id}: {bn:.2f}")
    except Exception:
        print(f"Background model attached but norm readout differed for OBS_ID {obs.obs_id}")

    # Incremental stack into the accumulator
    stacked.stack(dataset)

stacked


## Data inspection

Before fitting any source model, it is good practice to inspect the content of the dataset.  
We can look at the total counts map, the fitted background map, and the exposure map.  
This helps to verify that the FoV background method worked as expected and that the target source is clearly visible as an excess.


We can save this file so that later you can manipulate it!

```
stacked.write('stacked_MapDataset.fits')
```

We can also `peek` this, just like we did with the observations.

In [ ]:
stacked.write('stacked_MapDataset.fits',overwrite=True)
stacked.peek()

## Excess and significance maps (pre-model)

We compare **counts** vs. the **FoV background map** to build:
- **Excess map** = counts − background (integrated in energy)
- **Significance map** (Li & Ma–like, shown as `sqrt_ts`)

We use a **correlation radius** of `0.1 deg` and integrate from **0.8–20 TeV** (stable H.E.S.S. range).


In [ ]:
excess_estimator = ExcessMapEstimator(
    correlation_radius="0.1 deg",
    energy_edges=[0.4, 20]*u.TeV
)

excess_result = excess_estimator.run(stacked)

In [ ]:
excess_result.sqrt_ts.plot(add_cbar=True)
plt.show()

## 3D modeling — Gaussian spatial model with bounded position

We start from the target position (Galactic) and define a **GaussianSpatialModel**.  
To keep the fit stable, we **limit the source center** inside a **1° box** around the reference position (±0.5° in `l` and `b`).


In [ ]:
# Inspect the reference position
target_position

In [ ]:
spatial_model = GaussianSpatialModel(
    lon_0=target_position.l,
    lat_0=target_position.b,
    sigma=0.3 * u.deg,   # start slightly narrower; will be fit
    frame="galactic",
)

# Limit longitude within ±0.5° of the initial value
spatial_model.lon_0.min = spatial_model.lon_0.value - 0.5
spatial_model.lon_0.max = spatial_model.lon_0.value + 0.5

# Now do the same for latitude
spatial_model.lat_0.min = spatial_model.lat_0.value - 0.5
spatial_model.lat_0.max = spatial_model.lat_0.value + 0.5

spatial_model

## Spectral model and SkyModel

Create a **PowerLawSpectralModel** (index ≈ 2, amplitude ~ 1e-11 cm⁻² s⁻¹ TeV⁻¹ at 1 TeV)  
and combine with the Gaussian spatial model into a `SkyModel`.


In [ ]:
spectral_model = PowerLawSpectralModel(
    index=2.0,
    amplitude=1e-11 * u.Unit("cm-2 s-1 TeV-1"),
    reference=1 * u.TeV,
)

model_gauss = SkyModel(
    spatial_model=spatial_model,
    spectral_model=spectral_model,
    name="rxj1713_gauss",
)


## Add a background component to the models

The stacked dataset already has a *background map*, but to let the fit adjust it we attach a
`BackgroundModel` linked to the dataset name.


In [ ]:
bkg_model = FoVBackgroundModel(dataset_name=stacked.name)

In [ ]:
print(bkg_model)

## Attach models to the dataset 

In [ ]:
stacked.models = Models([model_gauss, bkg_model])
print(stacked.models)

- Let's check the dataset again

In [ ]:
print(stacked)
stacked.peek()

## Fit the current source model together with the FoV background


In [ ]:
fit = Fit()
result_1 = fit.run(datasets=[stacked])
print(result_1)

# Parameters (source + background)
stacked.models.to_parameters_table()

## Diagnostics after the fit

Integrated **counts / model (npred) / residuals**, and an **excess map** in 0.8–20 TeV.


In [ ]:
# Integrated maps
counts_int = stacked.counts.sum_over_axes()
model_int  = stacked.npred().sum_over_axes()
resid_int  = stacked.residuals().sum_over_axes()

fig, axes = plt.subplots(1, 3, figsize=(15, 4), subplot_kw={"projection": counts_int.geom.wcs})

counts_int.plot(ax=axes[0], add_cbar=True)
axes[0].set_title("Counts (integrated)")

model_int.plot(ax=axes[1], add_cbar=True)
axes[1].set_title("Model Npred (integrated)")

resid_int.plot(ax=axes[2], add_cbar=True)
axes[2].set_title("Residuals (sigma)")

plt.show()


In [ ]:
region = CircleSkyRegion(
    center=target_position, radius=1 * u.deg
)
stacked.plot_residuals(kwargs_spectral={"region":region});

## Source detection significance via nested model test 

We quantify **how significant** the source is above background using a **nested hypothesis test**:

- **H₀ (null):** background-only (the source flux is zero).  
- **H₁ (alt):** background + source (amplitude > 0).

We test the **`amplitude`** parameter of the source’s spectral model by fixing it to **0** under H₀ and fitting both models.  
The test returns a **Test Statistic (TS)**, which we convert to a **Gaussian significance** (σ) using `ts_to_sigma`.


In [ ]:
from gammapy.modeling.selection import select_nested_models
from gammapy.stats.utils import ts_to_sigma

# --- Freeze spatial parameters of the source model ---
model_gauss.spatial_model.lon_0.frozen = True
model_gauss.spatial_model.lat_0.frozen = True
model_gauss.spatial_model.sigma.frozen = True

# Parameter to test and its null value (H0)
parameters = [model_gauss.spectral_model.amplitude]
null_values = [0.0]   # hipótese nula: sem fluxo da fonte

results = select_nested_models(
    datasets=stacked,
    parameters=parameters,
    null_values=null_values,
    n_sigma=-np.inf,         # compute full TS (no truncation)
)

print(results['ts'])

ts = float(results["ts"])
sigma = ts_to_sigma(ts)
print(f"TS = {ts:.2f}  →  significance = {sigma:.2f} σ")
